# VII. XAI del Transformer (Estrategia Dual)

Este notebook implementa una estrategia de explicabilidad dual para el clasificador Transformer:

- **Intrínseca**: mapas de atención para ver en qué semanas/actividades se focaliza el modelo.
- **Agnóstica**: **SHAP** (KernelExplainer) para cuantificar contribución local/global de variables en la predicción de riesgo.

> Objetivo: reducir la caja negra y generar evidencia accionable para intervención temprana.

## 1) Setup y dependencias

In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score

sns.set_style('whitegrid')
np.set_printoptions(suppress=True, precision=4)

PROJECT_ROOT = Path('/workspace/TFM_education_ai_analytics')
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('TensorFlow =', tf.__version__)

In [ ]:
# Instala shap si falta (en este entorno no está instalado por defecto)
try:
    import shap
    print('shap version:', shap.__version__)
except Exception:
    !{sys.executable} -m pip install -q shap
    import shap
    print('shap version:', shap.__version__)

## 2) Configuración del experimento

In [ ]:
# Parámetros editables
UPTO_WEEK = 5
SPLIT = 'validation'   # training | validation | test
NUM_CLASSES = 2
WITH_STATIC = True

DATA_DIR = PROJECT_ROOT / 'data' / '6_transformer_features'
MODEL_PATH = PROJECT_ROOT / 'models' / 'transformers' / f'transformer_uptoW{UPTO_WEEK}_{NUM_CLASSES}clases.keras'
HISTORY_PATH = PROJECT_ROOT / 'reports' / 'transformer_training' / f'week_{UPTO_WEEK}' / f'experiments_history_{NUM_CLASSES}clases.json'

print('DATA_DIR   =', DATA_DIR)
print('MODEL_PATH =', MODEL_PATH)
print('HISTORY    =', HISTORY_PATH)

## 3) Carga de datos, modelo y utilidades

In [ ]:
import importlib.util

mod_path = PROJECT_ROOT / 'educational_ai_analytics' / '2_modeling' / 'transformers' / 'transformer_GLU_classifier.py'
spec = importlib.util.spec_from_file_location('transformer_GLU_classifier', mod_path)
tg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tg)

GLUTransformerClassifier = tg.GLUTransformerClassifier
TransformerEncoderBlock = tg.TransformerEncoderBlock
GLULayer = tg.GLULayer
PositionalEncoding = tg.PositionalEncoding

def load_npz_split(split: str, upto_week: int, with_static: bool = True):
    fp = DATA_DIR / split / f'transformer_uptoW{upto_week}.npz'
    d = np.load(fp, allow_pickle=True)

    X_seq = d['X_seq'].astype(np.float32)
    mask_pad = (d['mask_pad'] if 'mask_pad' in d.files else d['mask']).astype(np.int32)
    mask_activity = (d['mask_activity'] if 'mask_activity' in d.files else mask_pad).astype(np.int32)
    y = d['y'].astype(np.int64)
    ids = d['ids'].astype(str)

    X_static = None
    if with_static:
        X_static = d['X_static'].astype(np.float32)

    static_feature_names = d['static_feature_names'] if 'static_feature_names' in d.files else np.array([], dtype=object)
    activities = d['activities'] if 'activities' in d.files else np.array([], dtype=object)

    return {
        'X_seq': X_seq,
        'mask_pad': mask_pad,
        'mask_activity': mask_activity,
        'y': y,
        'ids': ids,
        'X_static': X_static,
        'static_feature_names': static_feature_names,
        'activities': activities,
    }


def load_model_from_history_and_weights(model_path: Path, history_path: Path, upto_week: int, x_seq_sample: np.ndarray, x_static_sample: np.ndarray | None):
    """
    Carga robusta para modelos subclassed:
    1) Lee hiperparámetros del historial (última corrida para la semana indicada).
    2) Construye arquitectura explícitamente.
    3) Carga pesos desde el archivo .keras con load_weights.
    """
    if not history_path.exists():
        raise FileNotFoundError(f"No existe historial: {history_path}")

    hist = json.loads(history_path.read_text(encoding='utf-8'))
    candidates = [r for r in hist if int(r.get('hyperparameters', {}).get('upto_week', -1)) == int(upto_week)]
    if not candidates:
        raise ValueError(f"No hay entradas en historial para upto_week={upto_week}")

    hp = candidates[-1]['hyperparameters']

    model = GLUTransformerClassifier(
        latent_d=int(hp.get('latent_d', 256)),
        num_heads=int(hp.get('num_heads', 8)),
        ff_dim=int(hp.get('ff_dim', 128)),
        dropout=float(hp.get('dropout', 0.3)),
        num_classes=int(hp.get('num_classes', 2)),
        num_layers=int(hp.get('num_layers', 2)),
        max_len=int(x_seq_sample.shape[1]),
        with_static_features=bool(hp.get('with_static', True)),
        static_hidden=tuple(hp.get('static_hidden', [64, 32])),
        head_hidden=tuple(hp.get('head_hidden', [256, 128])),
    )

    seq = x_seq_sample[:2].astype(np.float32)
    m = (seq.sum(axis=2) > 0).astype(np.int32)
    inputs = [seq, m, m]
    if bool(hp.get('with_static', True)) and x_static_sample is not None and x_static_sample.shape[1] > 0:
        inputs.append(x_static_sample[:2].astype(np.float32))

    _ = model(inputs, training=False)
    model.load_weights(model_path)
    return model

def filter_to_paper_binary(payload):
    # y original en tu pipeline: 0=Withdrawn, 1=Fail, 2=Pass, 3=Distinction
    # paper_baseline binario: remove Fail (1), y_bin=1 si Withdrawn(0), else 0
    y = payload['y']
    keep = y != 1

    X_seq = payload['X_seq'][keep]
    mask_pad = payload['mask_pad'][keep]
    mask_activity = payload['mask_activity'][keep]
    ids = payload['ids'][keep]
    X_static = payload['X_static'][keep] if payload['X_static'] is not None else None

    y_kept = y[keep]
    y_bin = np.where(y_kept == 0, 1, 0).astype(np.int64)

    return X_seq, mask_pad, mask_activity, X_static, y_bin, ids


def build_attention_mask(mask_pad, mask_activity):
    # mismo criterio reciente en entrenamiento: AND
    return (mask_pad.astype(np.int32) & mask_activity.astype(np.int32)).astype(np.int32)


payload = load_npz_split(SPLIT, UPTO_WEEK, with_static=WITH_STATIC)
X_seq, mask_pad, mask_activity, X_static, y_bin, ids = filter_to_paper_binary(payload)
attn_mask = build_attention_mask(mask_pad, mask_activity)

print('X_seq:', X_seq.shape, 'X_static:', None if X_static is None else X_static.shape)
print('y_bin:', y_bin.shape, 'positive_rate:', y_bin.mean().round(4))

model = load_model_from_history_and_weights(
    model_path=MODEL_PATH,
    history_path=HISTORY_PATH,
    upto_week=UPTO_WEEK,
    x_seq_sample=X_seq,
    x_static_sample=X_static,
)
print('Modelo cargado:', MODEL_PATH.name)

In [ ]:
# Predicción rápida para verificar que todo está alineado
inputs = [X_seq, attn_mask, mask_activity]
if X_static is not None:
    inputs.append(X_static)

probs = model.predict(inputs, verbose=0)
y_pred = np.argmax(probs, axis=1)

print('Balanced Acc:', round(balanced_accuracy_score(y_bin, y_pred), 4))
print('Confusion matrix:', confusion_matrix(y_bin, y_pred))
print(classification_report(y_bin, y_pred, digits=4))

## 4) XAI Intrínseca: Mapas de Atención

Extraemos scores de atención por capa/cabeza sin reentrenar el modelo.

In [ ]:
def forward_collect_attention_scores(model, x_seq, seq_mask, activity_mask, x_static=None):
    """
    Devuelve:
      - logits/probs finales
      - lista de attention scores por capa: [(B,H,W,W), ...]
    """
    x = model.input_proj(x_seq)

    x_static_emb = None
    if model.with_static_features:
        x_static_emb = model.static_block(x_static, training=False)
        gate = model.fusion_gate(x_static_emb)
        x = x + tf.expand_dims(gate * x_static_emb, axis=1)

    x = model.in_drop(x, training=False)

    layer_scores = []
    for enc in model.encoders:
        h = enc.norm_attn(x)
        attn_mask = enc.make_attn_mask(seq_mask)
        attn_out, attn_scores = enc.mha(
            h, h, h,
            attention_mask=attn_mask,
            return_attention_scores=True,
            training=False,
        )
        attn_out = enc.drop_attn(attn_out, training=False)
        x = x + attn_out

        h2 = enc.norm_ffn(x)
        ffn = enc.ffn_glu(h2)
        ffn = enc.ffn_out(ffn)
        ffn = enc.drop_ffn(ffn, training=False)
        x = x + ffn

        layer_scores.append(attn_scores.numpy())

    x = model.seq_out_norm(x)
    m = tf.cast(activity_mask, x.dtype)[:, :, tf.newaxis]
    avg_pooled = tf.reduce_sum(x * m, axis=1) / (tf.reduce_sum(m, axis=1) + 1e-8)
    x_masked = x + (1.0 - m) * -1e9
    max_pooled = tf.reduce_max(x_masked, axis=1)
    has_activity = tf.reduce_sum(m, axis=1) > 0
    max_pooled = tf.where(has_activity, max_pooled, tf.zeros_like(max_pooled))

    z = model.pooled_norm(tf.concat([avg_pooled, max_pooled], axis=-1))

    if model.with_static_features:
        x_static_raw_normed = model.static_raw_norm(x_static)
        z = tf.concat([z, x_static_emb, x_static_raw_normed], axis=1)

    probs = model.head(z, training=False).numpy()
    return probs, layer_scores


def plot_attention_heatmap(layer_scores, sample_idx=0, layer_idx=0, average_heads=True):
    scores = layer_scores[layer_idx][sample_idx]  # (H,W,W)
    if average_heads:
        mat = scores.mean(axis=0)
        title = f'Capa {layer_idx} | promedio cabezas'
    else:
        mat = scores[0]
        title = f'Capa {layer_idx} | cabeza 0'

    plt.figure(figsize=(6, 5))
    sns.heatmap(mat, cmap='magma')
    plt.title(title)
    plt.xlabel('Semana atendida (key)')
    plt.ylabel('Semana que atiende (query)')
    plt.tight_layout()
    plt.show()


probs_xai, attn_layers = forward_collect_attention_scores(
    model,
    X_seq.astype(np.float32),
    attn_mask.astype(np.int32),
    mask_activity.astype(np.int32),
    X_static.astype(np.float32) if X_static is not None else None,
)

print('N capas:', len(attn_layers), '| shape capa0:', attn_layers[0].shape)
plot_attention_heatmap(attn_layers, sample_idx=0, layer_idx=0, average_heads=True)

In [ ]:
# Ranking de semanas más "recibidas" por atención (métrica simple)
def temporal_focus_ranking(layer_scores, sample_idx=0, layer_idx=0):
    s = layer_scores[layer_idx][sample_idx].mean(axis=0)  # promedio cabezas -> (W,W)
    # importancia de semana j = suma de atención recibida como key
    received = s.sum(axis=0)
    df = pd.DataFrame({
        'week': np.arange(len(received)),
        'attention_received': received
    }).sort_values('attention_received', ascending=False)
    return df

focus_df = temporal_focus_ranking(attn_layers, sample_idx=0, layer_idx=0)
focus_df.head(10)

## 5) XAI Agnóstica: SHAP (KernelExplainer)

Para hacerlo tractable, explicamos la probabilidad de clase positiva (riesgo) con una muestra de fondo y una muestra de evaluación.

In [ ]:
# Construcción de matriz tabular explicable: [secuencia flatten + estáticas]
N, W, F = X_seq.shape
seq_dim = W * F

seq_feature_names = [f'w{w:02d}__{a}' for w in range(W) for a in payload['activities'].astype(str)]
static_feature_names = [str(c) for c in payload['static_feature_names']] if X_static is not None else []
all_feature_names = seq_feature_names + static_feature_names

X_flat_seq = X_seq.reshape(N, -1)
if X_static is not None:
    X_flat = np.concatenate([X_flat_seq, X_static], axis=1).astype(np.float32)
else:
    X_flat = X_flat_seq.astype(np.float32)

print('X_flat shape:', X_flat.shape)
print('n_features:', len(all_feature_names))

In [ ]:
# Wrapper para SHAP: convierte flat -> inputs del modelo
def predict_risk_proba_from_flat(X_flat_batch):
    X_flat_batch = np.asarray(X_flat_batch, dtype=np.float32)
    seq = X_flat_batch[:, :seq_dim].reshape(-1, W, F).astype(np.float32)

    # reconstruimos máscaras desde la secuencia perturbada
    activity = (seq.sum(axis=2) > 0).astype(np.int32)
    seq_mask = activity.copy()

    inputs_local = [seq, seq_mask, activity]

    if X_static is not None:
        stat = X_flat_batch[:, seq_dim:].astype(np.float32)
        inputs_local.append(stat)

    p = model.predict(inputs_local, verbose=0)
    # clase positiva = riesgo (índice 1 en binario del pipeline)
    return p[:, 1]


# Muestreo para costo computacional
rng = np.random.default_rng(42)
idx_bg = rng.choice(np.arange(N), size=min(80, N), replace=False)
idx_eval = rng.choice(np.setdiff1d(np.arange(N), idx_bg), size=min(40, max(1, N-len(idx_bg))), replace=False)

X_bg = X_flat[idx_bg]
X_eval = X_flat[idx_eval]

print('background:', X_bg.shape, '| eval:', X_eval.shape)

In [ ]:
# SHAP KernelExplainer (agnóstico al modelo)
import shap

explainer = shap.KernelExplainer(predict_risk_proba_from_flat, X_bg)
shap_values = explainer.shap_values(X_eval, nsamples=200)

# Compat para versiones SHAP
if isinstance(shap_values, list):
    shap_matrix = np.asarray(shap_values[0])
else:
    shap_matrix = np.asarray(shap_values)

print('shap_matrix:', shap_matrix.shape)

In [ ]:
# Importancia global (mean |SHAP|)
mean_abs = np.abs(shap_matrix).mean(axis=0)
global_imp = (
    pd.DataFrame({'feature': all_feature_names, 'mean_abs_shap': mean_abs})
    .sort_values('mean_abs_shap', ascending=False)
    .reset_index(drop=True)
)

global_imp.head(20)

In [ ]:
# Visualizaciones SHAP (global + local)
shap.summary_plot(shap_matrix, X_eval, feature_names=all_feature_names, max_display=20)

# Explicación local de un estudiante
local_i = 0
shap.force_plot(
    explainer.expected_value,
    shap_matrix[local_i],
    X_eval[local_i],
    feature_names=all_feature_names,
    matplotlib=True,
    figsize=(14, 3),
)

## 6) Lectura de resultados esperada

- **Atención**: semanas más atendidas por capa/cabeza -> evidencia temporal de foco del modelo.
- **SHAP global**: variables con mayor contribución media a riesgo/éxito.
- **SHAP local**: por estudiante, factores que empujan la predicción a riesgo o éxito.

Sugerencia metodológica:
1. Repetir para `UPTO_WEEK in [5,10,15]`.
3. Guardar tablas de `global_imp` para trazabilidad del TFM.